# NHS England Maternal Care Pathways Master Pipeline
## Stage 10 - Informative tables for paper

### Compiled by Federica Caretta Cortegiani

In [0]:
import sys
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql import SparkSession 
from pyspark.sql.functions import when, col, lit, count, max, last, first, min, avg, sum, collect_list, collect_set, countDistinct, to_date, datediff, array_contains, size, udf, format_number, regexp_replace, explode, array, array_union, desc_nulls_last, flatten, split, filter
from pyspark.sql.types import IntegerType, StringType, NumericType, DateType, ArrayType
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Imputer
from pyspark.ml.stat import Correlation, Summarizer
from pyspark.ml.functions import vector_to_array
from pyspark.sql.window import Window
from pyspark.ml.regression import LinearRegression
from pyspark.ml.regression import GeneralizedLinearRegression as GLR
import pyspark.pandas as ps
from functools import reduce

In [0]:
spark = SparkSession.builder.getOrCreate()

In [0]:
read_table_name = "msds_diag_busy_filtered_final_imputed_reduced_timed_ind_ranked_stage"

df_master = spark.read.table(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{read_table_name}")

In [0]:
year = [2021, 2022, 2023, 2024, 2025]
variables = ["Preeclampsia_during_this_pregnancy", "Preeclampsia_early_onset", "Preeclampsia_late_onset", "Possible_preeclampsia_diagnosis_AND1x_or_Hyp2x"]

pdf = pd.DataFrame(index=year, columns=variables)

for y in year:
    df = df_master.filter(col("birth_year") == y)
    for var in variables:
        pdf.at[y,var] = round((100 * df.filter(col(var) == 1).count() / df.filter(col(var).isNotNull()).count()), 2)

pdf = pdf.reset_index()
pdf.display()

In [0]:
model = 3
mixed_effect_cols = ["birth_year"] 
mixed_effect_cols += ["ld_hosp_org_site_id"]
mode = "a"
if len(mixed_effect_cols) == 2:
    mode = "b"

In [0]:
reg_dict = {
    "Num_obs": "N",
    "AIC (corrected)": "AIC",
    "aic": "AIC",
    "Intercept": "Intercept (base odds)",
    "age_at_booking": "Maternal Age",
    'booking_age_u35': "Under 35 (reference)", 
    'booking_age_35_40': "35-40", 
    'booking_age_o40': "Over 40",
    "estimated_bmi": "Maternal BMI",
    'mother_bmi_u185': "BMI < 18.5",
    'mother_bmi_185_249': "BMI 18.5 - 24.9 (reference)", 
    'mother_bmi_250_299': "BMI 25.0 - 29.9", 
    'mother_bmi_o300': "BMI > 30.0", 
    "ever_substance_use": "Ever Substance Use",
    "ever_smoker": "Ever Smoker",
    "disability_ind": "Disability",
    "folic_acid_while_pregnant": "Folic acid during pregnancy",
    "ethnic_white_reg": "White (Reference)",
    "ethnic_black_reg": "Black",
    "ethnic_south_asian_reg": "South Asian",
    "ethnic_mixed_reg": "Mixed ethnicity",
    "ethnic_other_reg": "Other ethnicity",
    "mother_unemployed": "Mother unemployed",
    "partner_unemployed": "Partner unemployed",
    "lang_not_english": "Language not English",
    "deprived_reg": "Socioeconomic deprivation (IMD <= 3)",
    "baby_male": "Male (Reference)",
    "baby_female": "Female",
    "previous_pregnancy": "Previous pregnancy",
    "prior_live_birth": "Previous live birth",
    "prior_csection": "Previous c-section",
    "prior_24w_loss": "Previous 24w loss",
    "prior_stillbirth": "Previous stillbirth",
    "Prior_Aborted_Pregnancy": "Previous aborted pregnancy",
    "contacts_attended": "Contacts attended (any mode)",
    "contacts_attended_pct": "Contacts attended percentage",
    'birth_2021': "Birth 2021 (Reference)",
    'birth_2022': "Birth 2022",
    'birth_2023': "Birth 2023",
    'birth_2024': "Birth 2024",
    'birth_2025': "Birth 2025",
    "Prior_Preeclampsia": "Previous preeclampsia",
    "Prior_Gestational_Diabetes": "Previous gestational DM",
    "Prior_Infectious_Disease": "Infectious Diseases",
    "Prior_Neoplasm": "Neoplasms",
    "Prior_Blood_or_Immune_Disease": "Blood or immune diseases",
    "Prior_Endocrine_or_Metabolic_Disease": "Endocrine/Metabolic Diseases",
    "Prior_Mental_Disorder": "Mental Disorders",
    "Prior_Nervous_System_Disease": "Nervous System Diseases",
    "Prior_Circulatory_Disease": "Circulatory Diseases",
    "Prior_Respiratory_Disease": "Respiratory Diseases",
    "Prior_Gastrointestinal_Disease": "Gastrointestinal Diseases",
    "Prior_Musculoskeletal_Disease": "Musculoskeletal Diseases",
    "Prior_Genitourinary_Disease": "Genitourinary Diseases",
    "Prior_Congenital_Abnormality": "Malformations/Abnormalities",
    "delivery_spont_ceph": "Spontaneous cephalic delivery (reference)",
    "delivery_any_csection": "C-Section delivery",
    "delivery_nonstandard_ceph": "Non-spontaneous cephalic delivery",
    "delivery_any_breech": "Breech delivery",
    "birth_weekday": "Birth Weekday (reference)",
    "birth_weekend": "Birth Weekend",
    "birth_winter": "Birth Winter (Jan-Mar) (Reference)",
    "birth_spring": "Birth Spring (Apr-Jun)",
    "birth_summer": "Birth Summer (Jul-Sep)",
    "birth_autumn": "Birth Autumn (Oct-Dec)",
    "birth_am": "Birth AM (reference)",
    "birth_pm": "Birth PM",
    'extremely_preterm_birth': "Extremely Preterm [< 28]", 
    'very_preterm_birth': "Very Preterm [28 - 31]", 
    'moderate_to_late_preterm_birth': "Moderate to Late Preterm [21 - 27]", 
    'not_preterm_birth' : "Not Preterm [>= 37] (reference)",
    'ld_spell_unit_busyness_on_arrival_3mo_p95_quint0': "Quintile 0",
    'ld_spell_unit_busyness_on_arrival_3mo_p95_quint1': "Quintile 1",
    'ld_spell_unit_busyness_on_arrival_3mo_p95_quint2': "Quintile 2 (Reference)",
    'ld_spell_unit_busyness_on_arrival_3mo_p95_quint3': "Quintile 3",
    'ld_spell_unit_busyness_on_arrival_3mo_p95_quint4': "Quintile 4",
    'ld_spell_unit_busyness_before_arrival_3mo_p95_quint0': "Quintile 0 prev",
    'ld_spell_unit_busyness_before_arrival_3mo_p95_quint1': "Quintile 1 prev",
    'ld_spell_unit_busyness_before_arrival_3mo_p95_quint2': "Quintile 2 prev (Reference)",
    'ld_spell_unit_busyness_before_arrival_3mo_p95_quint3': "Quintile 3 prev",
    'ld_spell_unit_busyness_before_arrival_3mo_p95_quint4': "Quintile 4 prev"
}

outcome_dict = {
    "any_preterm": "Any Preterm Birth",
    "major_preterm": "Preterm <32w",
    "birthweight": "Low birthweight",
    "preeclampsia": "Preeclampsia",
    "early_preeclampsia": "Early Preeclampsia (< 34 weeks)",
    "late_preeclampsia": "Late Preeclampsia (>= 34 weeks)",
    "late_preeclampsia_before_LD": "Late Preeclampsia (>= 34 weeks - <= LD+0)",
    "late_preeclampsia_after_LD": "Late Preeclampsia (>= 34 weeks - >= LD+1 & < LD discharge)",
    "gestDM": "Gestational DM",
    "early_gestDM": "Early Gestational DM (< 24 weeks)",
    "late_gestDM": "Late Gestational DM (>= 24 weeks)",
    "ecsection": "Emergency c-section",
    "agpar": "Low APGAR",
    "stillbirth": "Stillbirth",
    "readmit": "Readmission",
    "los": "L&D Log LOS (linear)",
    "neonatal_crit": "Neonatal Critical Incident",
    "nicu": "NICU",
    "neonatal_crit_filled": "Neonatal Critical Incident (filled)",
    "nicu_filled": "NICU (filled)" 
}

all_vars_dict = {
    'age_at_booking': "Maternal age",
    'booking_age_u35': "under 35 (reference)", 
    'booking_age_35_40': "35-40", 
    'booking_age_o40': "over 40",
    'estimated_bmi': "Maternal BMI",
    'mother_bmi_u185': "BMI < 18.5",
    'mother_bmi_185_249': "BMI 18.5 - 24.9 (reference)", 
    'mother_bmi_250_299': "BMI 25.0 - 29.9", 
    'mother_bmi_o300': "BMI > 30.0", 
    "ever_smoker": "Ever Smoker",
    "ever_smoking": "Ever Smoked",
    "folic_acid_while_pregnant": "Folic acid during pregnancy",
    "ethnic_white_reg": "White (Reference)",
    "ethnic_black_reg": "Black",
    "ethnic_south_asian_reg": "South Asian",
    "ethnic_mixed_reg": "Mixed ethnicity",
    "ethnic_other_reg": "Other ethnicity",
    "mother_unemployed": "Mother unemployed",
    "partner_unemployed": "Partner unemployed",
    "lang_not_english": "Language not English",
    "deprived_reg": "Socioeconomic deprivation (IMD <= 3)",
    "IMD_Rank_scaled": "Socioeconomic deprivation (IMD Rank, 0-10)",
    "contacts_attended": "Contacts attended (any mode)",
    "contacts_attended_pct": "Contacts attended percentage",
    "Prior_Preeclampsia": "Previous preeclampsia",
    "Prior_Gestational_Diabetes": "Previous gestational DM",
    "Prior_Endocrine_or_Metabolic_Disease": "Endocrine/Metabolic Diseases",
    "Prior_Mental_Disorder": "Mental Disorders",
    "Prior_Nervous_System_Disease": "Nervous System Diseases",
    "Prior_Circulatory_Disease": "Circulatory Diseases",
    "Prior_Respiratory_Disease": "Respiratory Diseases",
    "Prior_Gastrointestinal_Disease": "Gastrointestinal Diseases",
    "Prior_Musculoskeletal_Disease": "Musculoskeletal Diseases",
    "Prior_Congenital_Abnormality": "Malformations/Abnormalities",
    "birth_year": "Birth Year",
    "ld_hosp_org_site_id": "Hospital Site ID",
    "Preeclampsia_during_this_pregnancy": "Preeclampsia",
    "Preeclampsia_early_onset": "Early Preeclampsia (< 34 weeks)",
    "Preeclampsia_late_onset": "Late Preeclampsia (>= 34 weeks)",
}

In [0]:
reg_m1 = [
    #'deprived_reg'
    'IMD_Rank_scaled'
] 

reg_m1_ref = [
    #'deprived_reg'
    'IMD_Rank_scaled'
]

reg_m2 = reg_m1 + [
    # 'booking_age_u35', (reference)
    'booking_age_35_40', 
    'booking_age_o40',

    #'ethnic_white_reg', (reference)
    'ethnic_black_reg',
    'ethnic_south_asian_reg',
    'ethnic_mixed_reg',
    'ethnic_other_reg',

    'lang_not_english'
]

reg_m2_ref = reg_m1_ref + [
    'booking_age_u35',  # (reference)
    'booking_age_35_40', 
    'booking_age_o40',

    'ethnic_white_reg', #(reference)
    'ethnic_black_reg',
    'ethnic_south_asian_reg',
    'ethnic_mixed_reg',
    'ethnic_other_reg',

    'lang_not_english'
]

reg_m3 = reg_m2 + [
    'mother_bmi_u185', #underweight              
    # 'mother_bmi_185_249', #normal (reference) 
    'mother_bmi_250_299', #overweight 
    'mother_bmi_o300', #obese

    'ever_smoker',
    #'ever_smoking',
    'folic_acid_while_pregnant',   
    'contacts_attended_pct',
    #'contacts_attended',

    'Prior_Preeclampsia',
    'Prior_Gestational_Diabetes',
    'Prior_Endocrine_or_Metabolic_Disease',
    'Prior_Mental_Disorder',
    'Prior_Nervous_System_Disease',
    'Prior_Circulatory_Disease',
    'Prior_Respiratory_Disease',
    'Prior_Gastrointestinal_Disease',
    'Prior_Musculoskeletal_Disease',
    'Prior_Congenital_Abnormality'
]

reg_m3_ref = reg_m2_ref + [
    'mother_bmi_u185', #underweight               # influenced by IMD, but doesn't influence IMD
    'mother_bmi_185_249', #normal (reference) 
    'mother_bmi_250_299', #overweight 
    'mother_bmi_o300', #obese

    'ever_smoker',
    #'ever_smoking',
    'folic_acid_while_pregnant',
    'contacts_attended_pct',
    #'contacts_attended',

    'Prior_Preeclampsia',
    'Prior_Gestational_Diabetes',
    'Prior_Endocrine_or_Metabolic_Disease',
    'Prior_Mental_Disorder',
    'Prior_Nervous_System_Disease',
    'Prior_Circulatory_Disease',
    'Prior_Respiratory_Disease',
    'Prior_Gastrointestinal_Disease',
    'Prior_Musculoskeletal_Disease',
    'Prior_Congenital_Abnormality'
]

reg_vars = reg_m1
reg_vars_ref = reg_m1_ref
if model == 2:
    reg_vars = reg_m2
    reg_vars_ref = reg_m2_ref
elif model == 3:
    reg_vars = reg_m3
    reg_vars_ref = reg_m3_ref
reg_vars_1 = reg_vars.copy()
reg_vars_2 = reg_vars.copy()
if model == 3:
    reg_vars_1.remove("Prior_Preeclampsia")
    reg_vars_1.remove("Prior_Gestational_Diabetes")

In [0]:
outcome_vars = ["Preeclampsia_during_this_pregnancy", "Preeclampsia_early_onset", "Preeclampsia_late_onset"]

all_vars = reg_vars_ref + mixed_effect_cols + outcome_vars

## Missingness Table

In [0]:
df_select = df_master.select(all_vars)

df_missingness = df_select.agg(*[
    F.round(
        100 * F.sum(F.col(c).isNull().cast("int")) / F.count("*"),
        2
    ).alias(c)
    for c in df_select.columns
])

In [0]:
pdf = df_missingness.toPandas().T.reset_index()

pdf.columns = ["Variable", "Missingness"]

pdf["Variable"] = pdf["Variable"].map(all_vars_dict)

display(pdf) # Download directly from display

## Statistics Table

In [0]:
df_stats = df_master.withColumn("estimated_bmi", F.round((col("mother_avg_weight") / (lit(2.6244))), 1))
df_stats = df_stats.filter((col("group_2") == 1) |((col("group_1") == 1) & (~((col("Prior_Preeclampsia") == 1) | (col("Prior_Gestational_Diabetes") == 1)))))

df_group1 = df_stats.filter(col("group_1") == 1).filter(~((col("Prior_Preeclampsia") == 1) | (col("Prior_Gestational_Diabetes") == 1)))
df_group2 = df_stats.filter(col("group_2") == 1)

In [0]:
avg_vars = ["IMD_Rank_scaled", "age_at_booking", "estimated_bmi", "contacts_attended_pct"]
stat_vars = reg_vars_ref + outcome_vars
stat_vars.remove("IMD_Rank_scaled")
stat_vars.remove("contacts_attended_pct")

df_stats_all = (
    df_stats
    .agg(
        *[
            F.concat(
                F.sum(when(col(c) == 1, 1).otherwise(0)).cast("int").cast("string"),
                F.lit(" ("),
                F.round(100 * F.sum(when(col(c) == 1, 1).otherwise(0)) / F.count(col(c)), 1).cast("string"),
                F.lit("%)")
            ).alias(c)
            for c in stat_vars
        ] + [
            F.concat(
                F.round(F.mean(col(c)), 2).cast("string"),
                F.lit(" ("),
                F.round(F.stddev(col(c)), 2).cast("string"),
                F.lit(")")
            ).alias(c)
            for c in avg_vars
        ]
    )
)

df_stats_1 = (
    df_group1
    .agg(*[
        F.concat(
            F.sum(when(col(c) == 1, 1).otherwise(0)).cast("int").cast("string"),
            F.lit(" ("),
            F.round(100 * F.sum(when(col(c) == 1, 1).otherwise(0)) / F.count(col(c)), 1).cast("string"),
            F.lit("%)")
        ).cast("string").alias(c) 
        for c in stat_vars if c not in ["Prior_Preeclampsia", "Prior_Gestational_Diabetes"]
    ] + [
        F.lit(None) for c in ["Prior_Preeclampsia", "Prior_Gestational_Diabetes"]
    ] + [
        F.concat(
            F.round(F.mean(col(c)), 2).cast("string"),
            F.lit(" ("),
            F.round(F.stddev(col(c)), 2).cast("string"),
            F.lit(")")
        ).alias(c) 
        for c in avg_vars
    ])
)

df_stats_2 = (
    df_group2
    .agg(*[
        F.concat(
            F.sum(when(col(c) == 1, 1).otherwise(0)).cast("int").cast("string"),
            F.lit(" ("),
            F.round(100 * F.sum(when(col(c) == 1, 1).otherwise(0)) / F.count(col(c)), 1).cast("string"),
            F.lit("%)")
        ).cast("string").alias(c) for c in stat_vars
    ] + [
        F.concat(
            F.round(F.mean(col(c)), 2).cast("string"),
            F.lit(" ("),
            F.round(F.stddev(col(c)), 2).cast("string"),
            F.lit(")")
        ).alias(c)
        for c in avg_vars
    ])
)

In [0]:
var_order = [
    'IMD_Rank_scaled',

    'age_at_booking',
    'booking_age_u35',  # (reference)
    'booking_age_35_40', 
    'booking_age_o40',
    'ethnic_white_reg', #(reference)
    'ethnic_black_reg',
    'ethnic_south_asian_reg',
    'ethnic_mixed_reg',
    'ethnic_other_reg',
    'lang_not_english',

    'estimated_bmi',
    'mother_bmi_u185', #underweight               # influenced by IMD, but doesn't influence IMD
    'mother_bmi_185_249', #normal (reference) 
    'mother_bmi_250_299', #overweight 
    'mother_bmi_o300', #obese
    'ever_smoker',
    'folic_acid_while_pregnant',
    'contacts_attended_pct',

    'Prior_Preeclampsia',
    'Prior_Gestational_Diabetes',
    'Prior_Endocrine_or_Metabolic_Disease',
    'Prior_Mental_Disorder',
    'Prior_Nervous_System_Disease',
    'Prior_Circulatory_Disease',
    'Prior_Respiratory_Disease',
    'Prior_Gastrointestinal_Disease',
    'Prior_Musculoskeletal_Disease',
    'Prior_Congenital_Abnormality',

    "Preeclampsia_during_this_pregnancy", 
    "Preeclampsia_early_onset", 
    "Preeclampsia_late_onset"
]

In [0]:
def spark_to_stats_pdf(df, value_name):
    pdf = df.toPandas().T.reset_index()
    pdf.columns = ["Variable", value_name]
    pdf["Variable"] = pdf["Variable"].map(all_vars_dict)
    return pdf

pdf_all = spark_to_stats_pdf(df_stats_all, "All")
pdf_1 = spark_to_stats_pdf(df_stats_1, f"Nulliparous (n={nulliparous})")
pdf_2 = spark_to_stats_pdf(df_stats_2, f"Multiparous (n={parous})")

pdf_stats = reduce(
    lambda left, right: left.merge(right, on="Variable", how="outer"),
    [pdf_all, pdf_1, pdf_2]
)

pdf_stats["Variable"] = pd.Categorical(
    pdf_stats["Variable"],
    categories=[all_vars_dict[v] for v in var_order],
    ordered=True
)

pdf_stats = pdf_stats.sort_values("Variable").reset_index(drop=True)

display(pdf_stats)  # Download directly from display